# ViT-LENS: Scientific Backend Parity Audit (Colab GPU)

**Research Objective**: Independently audit and benchmark the scientific correctness of the Python backend against direct standalone research formulas using the real trained TopK SAE checkpoint (`final_sae_best.pt`).

---

### Two Independent Execution Paths Compared:
1. **Path A (Direct Research Implementation)**: Standalone, direct evaluation of the tested research equations without wrapper indirection.
2. **Path B (Production Backend)**: The modular `ModelService`, `ViTModelWrapper`, `FeatureLocalizer`, and `ResidualPreservingIntervention` used by the application.

### Tested Quantities (Numerical Tolerance $\le 10^{-5}$):
- Preprocessed input tensors
- Pristine ViT 1000-class logits & probabilities
- Full SAE latent activation tensor $[196, 3072]$
- Selected feature 196 patch activations
- 14×14 spatial activation matrix
- Peak activation, active patch count, and argmax row/column
- Intervened logits at $\alpha = 0.0$ (Causal Suppression)
- Identity logits at $\alpha = 1.0$ (Must equal original logits $< 10^{-6}$)
- Intervened logits at $\alpha = 2.0$ (Causal Amplification)
- Target class logit shifts and probability deltas

### Step 1: Mount Google Drive & Environment Setup

In [ ]:
import os
import sys

# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Mounted Google Drive successfully.")
except ImportError:
    print("Not in Google Colab; skipping Drive mount.")

# 2. Locate and set repository root
candidates = [
    "/content/ViT-LENS",
    "/content/drive/MyDrive/ViT-LENS",
    os.path.abspath(os.path.join(os.getcwd(), "..")),
    os.getcwd()
]
repo_root = None
for c in candidates:
    if os.path.exists(os.path.join(c, "src", "topk_sae.py")):
        repo_root = c
        break

if repo_root:
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    os.chdir(repo_root)
    print(f"Active repository root: {repo_root}")
else:
    print("Warning: Repository root not found in standard paths. Current CWD:", os.getcwd())

### Step 2: GPU Hardware & Parameter Device Verification

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Hardware Acceleration: CUDA GPU ({gpu_name}, {vram_gb:.2f} GB VRAM)")
else:
    device = torch.device("cpu")
    print("Hardware Acceleration: CPU (Warning: For optimal demonstration speed, switch Colab runtime to T4 GPU)")

### Step 3: Install Required Dependencies

In [ ]:
!pip install -q timm gradio matplotlib pillow pyyaml

### Step 4: Validate Checkpoint & Load Pretrained ViT-B/16

In [ ]:
from src.utils import resolve_sae_checkpoint_path
from src.vit_model import ViTModelWrapper
from src.topk_sae import TopKSparseAutoencoder

ckpt_path = resolve_sae_checkpoint_path()
print(f"Target SAE Checkpoint Path: {ckpt_path}")
print(f"Checkpoint File Exists: {os.path.exists(ckpt_path)}")

# Load ViT-B/16
vit = ViTModelWrapper(
    model_name="vit_base_patch16_224.augreg_in1k",
    pretrained=True,
    target_layer_idx=7,
    device=device
)
print(f"\nViT Model Loaded: {vit.model_name}")
print(f"ViT Parameter Device: {vit.vit_parameter_device}")
print(f"Resolved Data Configuration: {vit.get_data_config()}")

# Load SAE Checkpoint
if os.path.exists(ckpt_path):
    print(f"\nLoading validated SAE checkpoint from {ckpt_path}...")
    sae = TopKSparseAutoencoder.load_from_checkpoint(ckpt_path, device=device)
    print("SAE Loaded Successfully from Checkpoint: YES")
    print(f"SAE Parameter Device: {next(sae.parameters()).device}")
    print(f"SAE Dimensions: d_in={sae.d_in}, d_sae={sae.d_sae}, top_k={sae.top_k}")
else:
    raise FileNotFoundError(
        f"Trained SAE checkpoint not found at: {ckpt_path}\n"
        "Please mount Google Drive at /content/drive to access your trained final_sae_best.pt,\n"
        "or set the environment variable VIT_LENS_SAE_CHECKPOINT to the exact path."
    )

### Step 5: Execute Scientific Backend Parity Audit
Runs the complete numerical parity audit comparing Path A (Direct Research Formulas) against Path B (Production ModelService).

In [ ]:
from scripts.backend_parity_audit import run_case_parity_audit
from src.model_service import ModelService

# Initialize production service
service = ModelService(vit=vit, sae=sae, device=device)

# Images and features to audit
samples_dir = os.path.join(repo_root or ".", "data", "samples")
fish_img = os.path.join(samples_dir, "sample_fish.png")
springer_img = os.path.join(samples_dir, "sample_springer.png")
retriever_img = os.path.join(samples_dir, "sample_retriever.png")

audit_cases = [
    ("Case 1: Tench Image / Feature #588 (Class 0: Tench)", fish_img, 588, 0),
    ("Case 2: English Springer / Feature #1453 (Class 217: English Springer)", springer_img, 1453, 217),
    ("Case 3: Golden Retriever / Discovered Feature (Class 207: Retriever)", retriever_img, 2567, 207),
    ("Case 4: Inactive Feature (Feature #1453 on Tench Image)", fish_img, 1453, 0),
]

case_results = []
for name, img_path, fid, target_cls in audit_cases:
    if os.path.exists(img_path):
        res = run_case_parity_audit(
            case_name=name,
            image_path=img_path,
            feature_id=fid,
            target_class_idx=target_cls,
            vit_wrapper=vit,
            sae=sae,
            service=service,
            tol=1e-5
        )
        case_results.append(res)
    else:
        print(f"Warning: Image not found: {img_path}")

all_passed = all(r["passed"] for r in case_results)
print("\n" + "="*75)
print(f"FINAL AUDIT RESULT: {'ALL PARITY TESTS PASSED [OK]' if all_passed else 'PARITY FAILURES DETECTED [FAIL]'}")
print("="*75)